In [28]:
import h5py
import pandas as pd
import numpy as np
import os

# ==========================================
# 1. P 波到達時間查詢器 (負責跨表比對 CSV)
# ==========================================
class PWaveArrivalLocator:
    def __init__(self, csv_path):
        print(f"[*] 正在初始化並載入 CSV 檔案: {csv_path} ...")
        self.df = pd.read_csv(csv_path, low_memory=False)
        
        required_cols = ['source_event_id', 'station_code', 'trace_p_arrival_sample', 'trace_snr_db', 'pga']
        for col in required_cols:
            if col not in self.df.columns:
                raise ValueError(f"[!] CSV 檔案中缺少必要的欄位: {col}")
                
        # 強制轉為字串，確保後續 startswith 可以正常運作
        self.df['source_event_id'] = self.df['source_event_id'].astype(str)
        print("[+] CSV 載入完成，查詢器準備就緒！\n")

    def get_trace_info(self, hdf5_key):
        try:
            parts = hdf5_key.split('_')
            if len(parts) < 3:
                return None
                
            event_id = parts[1]      
            station_code = parts[2]  
        except Exception:
            return None

        condition = (self.df['source_event_id'].str.startswith(event_id)) & \
                    (self.df['station_code'] == station_code)
                    
        matched_row = self.df[condition]
        
        if not matched_row.empty:
            # 取得各欄位的數值
            snr = matched_row['trace_snr_db'].iloc[0]
            arrival_sample = matched_row['trace_p_arrival_sample'].iloc[0]
            pga_value = matched_row['pga'].iloc[0]
            
            # --- 修改 2: 條件過濾 (SNR <= 10 或是空值就跳過) ---
            if pd.isna(snr) or snr <= 10:
                return None
                
            # 確保提取的目標值不是空值
            if pd.isna(arrival_sample) or pd.isna(pga_value):
                return None
                
            # --- 修改 3: 打包回傳多個值 ---
            return {
                'p_arrival': int(arrival_sample),
                'pga': float(pga_value)
            }
        
        else:
            return None



# ==========================================
# 2. HDF5 波形裁切器 (支援 PGA 提取與 SNR 過濾)
# ==========================================

def create_p_wave_dataset(hdf5_path, locator, limit=3000):
    import os
    if not os.path.exists(hdf5_path):
        print(f"[!] 錯誤：找不到 HDF5 檔案 {hdf5_path}")
        return None

    final_dataset = {}
    
    # --- 新增總檢查計數器 ---
    total_checked = 0  
    success_count = 0
    skip_csv_or_snr = 0  
    skip_length_error = 0
    
    print(f"[*] 準備開啟 HDF5 檔案: {hdf5_path}")
    with h5py.File(hdf5_path, 'r') as f:
        hdf5_keys = list(f.keys())
        print(f"[+] HDF5 內共有 {len(hdf5_keys)} 筆資料，開始進行比對與裁切...")
        
        for key in hdf5_keys:
            # 當成功收集的數量達到目標時，停止迴圈
            if success_count >= limit:
                break

            # --- 只要一進入迴圈看這筆資料，檢查總數就 +1 ---
            total_checked += 1  

            trace_info = locator.get_trace_info(key)
            
            if trace_info is not None:
                p_arrival = trace_info['p_arrival']
                pga_value = trace_info['pga']
                
                try:
                    waveform = f[key][:] 
                except Exception as e:
                    print(f"  [!] 無法讀取 {key} 的波形: {e}")
                    continue
                
                start_idx = p_arrival - 100
                end_idx = p_arrival + 300
                
                if len(waveform.shape) >= 2:
                    time_length = waveform.shape[1] 
                    
                    if start_idx >= 0 and end_idx <= time_length:
                        sliced_wave = waveform[:, start_idx:end_idx] 
                        
                        final_dataset[key] = {
                            'key': key,
                            'trace_p_arrival_sample': p_arrival,
                            'pga': pga_value,             
                            'wave_data': sliced_wave.tolist() 
                        }
                        
                        success_count += 1
                    else:
                        skip_length_error += 1
                else:
                    skip_length_error += 1
            else:
                skip_csv_or_snr += 1
                
    # ==========================================
    # 更新：更精確且不會讓人誤會的輸出訊息
    # ==========================================
    print("\n" + "=" * 50)
    print(f"✅ 任務結束！目標收集 {limit} 筆，實際成功收集 {success_count} 筆。")
    print(f"🔍 為了達成目標，系統在 HDF5 中總共檢查了: {total_checked} 筆資料")
    print("-" * 50)
    print(f"📊 [成功收編] 乾淨且長度合格: {success_count} 筆")
    print(f"⚠️ [中途淘汰] CSV 找不到或 SNR <= 10: {skip_csv_or_snr} 筆")
    print(f"⚠️ [中途淘汰] 波形長度不足或邊界溢出: {skip_length_error} 筆")
    print("=" * 50)
    
    return final_dataset


# 存成新的檔案
import numpy as np

def save_dataset_to_numpy(dataset_dict, output_filename='3000_p_wave_dataset.npz'):
    """
    將整理好的字典拆解並儲存為高壓縮率的 NumPy .npz 檔案 (包含新增的 PGA)
    """
    print(f"\n[*] 準備將 {len(dataset_dict)} 筆資料轉換為 NumPy 陣列...")
    
    # 1. 準備空列表來收集對應的資料
    keys_list = []
    p_arrivals_list = []
    pga_list = []          # 新增：收集 PGA 數值
    waveforms_list = []
    
    # 2. 遍歷字典，把資料分門別類裝好
    for dict_key, data in dataset_dict.items():
        keys_list.append(data['key'])
        p_arrivals_list.append(data['trace_p_arrival_sample'])
        pga_list.append(data['pga'])         # 新增：提取 PGA
        waveforms_list.append(data['wave_data'])
        
    # 3. 轉換為 NumPy Array
    keys_array = np.array(keys_list)
    p_arrivals_array = np.array(p_arrivals_list)
    pga_array = np.array(pga_list, dtype=np.float32)  # 新增：轉換為 NumPy 陣列 (浮點數)
    
    # 波形轉換後會變成強大的三維陣列 (N, 3, 400)
    waveforms_array = np.array(waveforms_list, dtype=np.float32) 
    
    print(f"[*] 轉換完成！波形陣列形狀為: {waveforms_array.shape}")
    print(f"[*] PGA 標籤陣列形狀為: {pga_array.shape}")
    
    # 4. 打包並壓縮儲存
    # 將 pgas=pga_array 一併存入壓縮包中
    np.savez_compressed(
        output_filename, 
        keys=keys_array, 
        p_arrivals=p_arrivals_array, 
        pgas=pga_array,          # 新增：儲存 PGA
        waveforms=waveforms_array
    )
    print(f"✅ 成功儲存至: {output_filename}")

# ==========================================
# 讀取測試 (驗證剛才存進去的資料沒壞掉)
# ==========================================
def load_and_test_numpy(filename='3000_p_wave_dataset.npz'):
    print(f"\n[*] 正在讀取測試 {filename} ...")
    
    # 使用 allow_pickle=True 是因為 keys 裡面裝的是字串物件
    with np.load(filename, allow_pickle=True) as data:
        loaded_keys = data['keys']
        loaded_arrivals = data['p_arrivals']
        loaded_pgas = data['pgas']           # 新增：讀取 PGA
        loaded_waves = data['waveforms']
        
        print(f"[+] 讀取成功！")
        print(f"   - 總資料筆數: {len(loaded_keys)}")
        print(f"   - 波形總形狀: {loaded_waves.shape}  <-- (資料筆數, 通道數, 採樣點)")
        print(f"   - PGA 總形狀: {loaded_pgas.shape}")
        
        # 印出第一筆資料來確認
        if len(loaded_keys) > 0:
            print(f"\n[檢查] 第一筆資料還原結果：")
            print(f"   - Key: {loaded_keys[0]}")
            print(f"   - P波位置: {loaded_arrivals[0]}")
            print(f"   - PGA 數值: {loaded_pgas[0]}")  # 新增：印出 PGA 檢查
            print(f"   - 第 0 通道前 5 個點: {loaded_waves[0][0][:5]}")


# ==========================================
# 3. 執行主程式
# ==========================================
if __name__ == "__main__":
    # --- 請修改為你 Mac 本機的實際路徑 ---
    csv_file = './all_metadata.csv' 
    hdf5_file = './sample_3000.hdf5'          
    
    try:
        # 1. 實例化查詢器
        my_locator = PWaveArrivalLocator(csv_file)
        
        # 2. 執行裁切任務 (設為 100 筆做快速測試)
        my_dataset = create_p_wave_dataset(hdf5_file, my_locator, limit=3000)
        
        # 3. 檢查產出的字典結構
        if my_dataset and len(my_dataset) > 0:
            print("\n[檢查] 擷取到的前 2 筆資料結構：")
            for i, dict_key in enumerate(list(my_dataset.keys())[:2], 1):
                data = my_dataset[dict_key]
                print(f"\n{i}. 外部字典 Key: {dict_key}")
                print(f"   - 內部儲存的 Key: {data['key']}")
                print(f"   - P波位置: {data['trace_p_arrival_sample']}")
                print(f"   - PGA 標籤值 (Target): {data['pga']}")
                print(f"   - 波形通道數 (Channels): {len(data['wave_data'])} 通道")
                print(f"   - 單一通道波形長度: {len(data['wave_data'][0])} 點")

        if my_dataset and len(my_dataset) > 0:
            # 1. 執行儲存 (存成你指定的檔名)
            save_dataset_to_numpy(my_dataset, output_filename='3000_p_wave_dataset.npz')
            
            # 2. 執行讀取測試 (驗證)
            load_and_test_numpy('3000_p_wave_dataset.npz')
                
    except Exception as e:
        print(f"\n[!] 程式執行發生錯誤: {e}")


[*] 正在初始化並載入 CSV 檔案: ./all_metadata.csv ...
[+] CSV 載入完成，查詢器準備就緒！

[*] 準備開啟 HDF5 檔案: ./sample_3000.hdf5
[+] HDF5 內共有 3000 筆資料，開始進行比對與裁切...

✅ 任務結束！目標收集 3000 筆，實際成功收集 1011 筆。
🔍 為了達成目標，系統在 HDF5 中總共檢查了: 3000 筆資料
--------------------------------------------------
📊 [成功收編] 乾淨且長度合格: 1011 筆
⚠️ [中途淘汰] CSV 找不到或 SNR <= 10: 1989 筆
⚠️ [中途淘汰] 波形長度不足或邊界溢出: 0 筆

[檢查] 擷取到的前 2 筆資料結構：

1. 外部字典 Key: 2012_1201010230_ALS_HL_SMT_10
   - 內部儲存的 Key: 2012_1201010230_ALS_HL_SMT_10
   - P波位置: 2747
   - PGA 標籤值 (Target): 0.116434656
   - 波形通道數 (Channels): 3 通道
   - 單一通道波形長度: 400 點

2. 外部字典 Key: 2012_1201010230_CHK_HL_BB_10
   - 內部儲存的 Key: 2012_1201010230_CHK_HL_BB_10
   - P波位置: 2353
   - PGA 標籤值 (Target): 0.431441443
   - 波形通道數 (Channels): 3 通道
   - 單一通道波形長度: 400 點

[*] 準備將 1011 筆資料轉換為 NumPy 陣列...
[*] 轉換完成！波形陣列形狀為: (1011, 3, 400)
[*] PGA 標籤陣列形狀為: (1011,)
✅ 成功儲存至: 3000_p_wave_dataset.npz

[*] 正在讀取測試 3000_p_wave_dataset.npz ...
[+] 讀取成功！
   - 總資料筆數: 1011
   - 波形總形狀: (1011, 3, 400)  <-- (資料筆數, 通道數, 採樣點)
   - PGA 總形狀

# 檢查結構

In [27]:
import numpy as np

def inspect_npz_structure(file_path):
    print(f"[*] 正在讀取檔案: {file_path}")
    try:
        # 載入 .npz 檔案，設定 allow_pickle=True 是為了能讀取裝有字串的 keys 陣列
        data = np.load(file_path, allow_pickle=True)
        
        print("\n[+] 檔案載入成功！包含以下陣列：")
        print("=" * 50)
        
        # data.files 會列出這個壓縮包裡面所有的陣列名稱
        for array_name in data.files:
            arr = data[array_name]
            print(f"📦 陣列名稱: '{array_name}'")
            print(f"   - 形狀 (Shape): {arr.shape}")
            print(f"   - 資料型態 (dtype): {arr.dtype}")
            
            # 印出第一筆資料作為範例來確認內容
            if len(arr) > 0:
                if array_name == 'waveforms':
                    # 波形陣列太大，我們只印出第一筆的形狀，以及第 0 通道的前 5 個點
                    print(f"   - 第一筆範例: {arr[0].shape} 的矩陣 (第 0 通道前 5 個點: {arr[0][0][:5]})")
                else:
                    print(f"   - 第一筆範例: {arr[0]}")
            print("-" * 50)
            
    except FileNotFoundError:
        print(f"[!] 錯誤：找不到檔案 '{file_path}'，請確認路徑是否正確。")
    except Exception as e:
        print(f"[!] 發生未知的錯誤: {e}")

if __name__ == "__main__":
    # 替換成你剛剛存好的檔案名稱
    file_path = "1000_p_wave_dataset.npz" 
    inspect_npz_structure(file_path)

[*] 正在讀取檔案: 1000_p_wave_dataset.npz

[+] 檔案載入成功！包含以下陣列：
📦 陣列名稱: 'keys'
   - 形狀 (Shape): (364,)
   - 資料型態 (dtype): <U30
   - 第一筆範例: 2012_1201010230_ALS_HL_SMT_10
--------------------------------------------------
📦 陣列名稱: 'p_arrivals'
   - 形狀 (Shape): (364,)
   - 資料型態 (dtype): int64
   - 第一筆範例: 2747
--------------------------------------------------
📦 陣列名稱: 'pgas'
   - 形狀 (Shape): (364,)
   - 資料型態 (dtype): float32
   - 第一筆範例: 0.11643465608358383
--------------------------------------------------
📦 陣列名稱: 'waveforms'
   - 形狀 (Shape): (364, 3, 400)
   - 資料型態 (dtype): float32
   - 第一筆範例: (3, 400) 的矩陣 (第 0 通道前 5 個點: [0.0007335  0.00067642 0.00053669 0.00036293 0.00020969])
--------------------------------------------------
